# Aula 5, IA generativa

Notebook executável que acompanha a aula [05-ia-generativa.md](../../lessons/modulo-01-introducao-ia/05-ia-generativa.md).

## O que vamos fazer aqui

Vamos construir do zero um gerador de texto por n-gramas, que cria frases novas
prevendo a próxima palavra a partir das anteriores. É a mesma intuição dos grandes
modelos de linguagem, só que em miniatura.

Primeiro fazemos um gerador de bigramas, que olha só a palavra anterior. Depois, no
projeto, um de trigramas, que olha as duas anteriores e soa mais coerente. Por fim,
comparamos com um modelo de verdade via Ollama, para ver o que muda com escala. A
parte dos n-gramas é Python puro, e a do Ollama degrada de forma graciosa.

## Gerador de bigramas

Treinar um bigrama é guardar, para cada palavra, a lista de palavras que a seguiram
no texto. Gerar é começar de uma palavra e ir sorteando a próxima a partir dessa
lista. Como palavras mais comuns aparecem mais vezes na lista, o sorteio já respeita
a frequência observada.

In [ ]:
import random
from collections import defaultdict

# Pequeno corpus de treino, em minúsculas e separado por espaços.
corpus = (
    'o aluno estuda matemática todos os dias . '
    'o aluno resolve exercícios de matemática . '
    'a professora explica a matéria com calma . '
    'a professora corrige os exercícios do aluno . '
    'o aluno aprende matemática com a professora .'
).split()


def treinar_bigramas(tokens):
    """Para cada palavra, guarda a lista de palavras que a seguem no corpus."""
    modelo = defaultdict(list)
    for atual, proximo in zip(tokens, tokens[1:]):
        modelo[atual].append(proximo)
    return modelo


def gerar(modelo, inicio, tamanho=12):
    """Gera texto amostrando a próxima palavra a partir da anterior."""
    palavra = inicio
    saida = [palavra]
    for _ in range(tamanho):
        proximas = modelo.get(palavra)
        if not proximas:
            break
        palavra = random.choice(proximas)
        saida.append(palavra)
    return ' '.join(saida)


random.seed(0)
modelo_bi = treinar_bigramas(corpus)
for _ in range(3):
    print(gerar(modelo_bi, 'o'))

As frases geradas fazem sentido de palavra para palavra, mas se perdem no
sentido global, porque o modelo só lembra da palavra anterior. Essa limitação de
memória curta é justamente o que os Transformers vão resolver, olhando um contexto
amplo de uma vez.

## Projeto da aula, bigramas contra trigramas

No trigrama, a chave passa a ser o par das duas palavras anteriores. Com mais
contexto, o texto tende a ficar mais coerente, ao custo de precisar de mais dados.
Vamos comparar os dois geradores.

In [ ]:
def treinar_trigramas(tokens):
    modelo = defaultdict(list)
    for a, b, proximo in zip(tokens, tokens[1:], tokens[2:]):
        modelo[(a, b)].append(proximo)
    return modelo


def gerar_tri(modelo, inicio, tamanho=12):
    a, b = inicio
    saida = [a, b]
    for _ in range(tamanho):
        proximas = modelo.get((a, b))
        if not proximas:
            break
        proxima = random.choice(proximas)
        saida.append(proxima)
        a, b = b, proxima
    return ' '.join(saida)


random.seed(0)
modelo_tri = treinar_trigramas(corpus)
print('Bigramas :', gerar(modelo_bi, 'o'))
print('Trigramas:', gerar_tri(modelo_tri, ('o', 'aluno')))

O trigrama segue trechos mais longos do corpus original e soa mais coerente.
Com um corpus pequeno como este, porém, ele também repete mais o texto de treino, o
que mostra a necessidade de mais dados. Para o projeto, gere algumas frases com cada
um e explique por que o trigrama tende a soar melhor.

## Comparando com um modelo de verdade via Ollama

A mesma ideia de prever a próxima palavra, levada à escala, produz texto muito mais
rico. A célula abaixo pede a um modelo local que escreva uma explicação curta para um
aluno.

In [ ]:
try:
    import ollama

    resposta = ollama.chat(
        model='llama3.1',
        messages=[{'role': 'user', 'content': (
            'Escreva, em duas frases, uma explicação simples de o que é uma função '
            'matemática, para um estudante do primeiro ano.'
        )}],
    )
    print(resposta['message']['content'])
except Exception as erro:
    print('Não foi possível usar o Ollama agora.')
    print('Veja docs/setup.md. Detalhe técnico:', erro)

A diferença de qualidade entre o gerador de n-gramas e o modelo via Ollama
vem sobretudo de quanto contexto cada um considera e de quanto texto cada um viu no
treino. Guarde essa observação, pois ela motiva os Transformers e os LLMs que
estudaremos mais adiante na trilha.